# Controlled nanoGPT tuning and one measured AdamW update

This third experiment preserves both original course experiments. It tests learning rate and passage sampling with the expanded corpus, vocabulary, split and architecture fixed.

**Prediction recorded before execution:** balanced sampling may improve grammar and contrasts, with a risk of weakening starter patterns. Confidence is moderate in this hypothesis; benchmark improvement is unknown. Existing vocabulary covers 27 of 48 cases.

Six configurations combine peak learning rates 0.0003, 0.001 and 0.003 with uniform sampling or batches of 16 starter, eight grammar and eight contrast passages. All use 3,000 updates and batch size 32.

Select by final mean category validation loss, subject to original-panel loss increasing at most 5%. Confirm the selected configuration against baseline with seeds 43 and 44. The public suite is evaluated after selection and confirmation. See the companion source for the complete training loop.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nanogpt_model.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.tune_controlled import Study
output = ROOT / 'tuning_runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
study = Study(output)
print('Study:', output.relative_to(ROOT))

Frozen inputs and panels; training categories: {'starter': 4148, 'grammar': 464, 'contrasts': 341}
Six configurations; final checkpoint selection precedes language evaluation.
Study: tuning_runs/20260923T065357_257422Z


## Six configurations
All settings for a given seed start from identical weights. Within each sampling method, every learning rate receives the same sequence of minibatches. Seed 42 must exactly reproduce the saved original baseline.

In [2]:
study.sweep()

uniform_lr0.001_seed42: macro=0.906704, original=0.746731, 15.6s


uniform_lr0.0003_seed42: macro=1.216935, original=0.838709, 14.1s


uniform_lr0.003_seed42: macro=0.913143, original=0.726566, 16.2s


balanced_lr0.0003_seed42: macro=0.981257, original=0.874261, 19.6s


balanced_lr0.001_seed42: macro=0.893670, original=0.749250, 17.7s


balanced_lr0.003_seed42: macro=0.834300, original=0.755719, 19.2s


Baseline exactly reproduces original expanded model weights.


## Select using validation
The panels were frozen before training: the original 20 passages and a second panel of 10 starter, five grammar and five contrast passages. Each category loss averages non-padding next-token targets; the selection metric gives categories equal weight.

In [3]:
selection = study.select()

Selected using validation only: {'selected_utc': '2026-09-23T06:55:41.782029+00:00', 'selected_run': 'balanced_lr0.003_seed42', 'configuration': ['balanced', 0.003], 'original_panel_threshold': 0.7840672194957733, 'eligible': ['uniform_lr0.001_seed42', 'uniform_lr0.003_seed42', 'balanced_lr0.001_seed42', 'balanced_lr0.003_seed42'], 'language_evaluation_used': False, 'selection_metric': 'final macro category validation NLL'}


## Confirm across seeds and check instrumentation
Repeat baseline and selected settings for two additional initialization and sampling seeds, keeping the data split fixed. Repeat selected seed 42 without update instrumentation and require exact equality of final weights, optimizer state and RNG state.

In [4]:
study.confirm()

uniform_lr0.001_seed43: macro=0.950066, original=0.740955, 17.0s


balanced_lr0.003_seed43: macro=0.886496, original=0.748076, 17.8s


uniform_lr0.001_seed44: macro=0.910644, original=0.741713, 16.2s


balanced_lr0.003_seed44: macro=0.850648, original=0.756761, 17.3s


balanced_lr0.003_seed42_control: macro=0.834300, original=0.755719, 17.3s


Instrumented and uninstrumented final model, optimizer and RNG are exactly equal.


## Evaluate saved models
Run the unchanged 48 cases on all final checkpoints and the three untrained initializations. Retain every case, CSV, summary and free continuation. The scores below cannot influence the already saved selection.

In [5]:
pairs = study.evaluate()
from IPython.display import display, Markdown
import json
display(Markdown('| Seed | Baseline correct | Selected correct | Macro loss change | Original guardrail |\n|---|---:|---:|---:|---|\n' + '\n'.join(f"| {p['seed']} | {p['baseline_correct']}/48 | {p['selected_correct']}/48 | {p['macro_loss_change']:+.6f} | {p['original_guardrail_pass']} |" for p in pairs)))

Language evals (untrained): 9/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 2/24 correct; 3 scorable
  starter_patterns: 4/16 correct; 16 scorable
  starter_transfer: 3/8 correct; 8 scorable


Language evals (untrained): 10/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 0/24 correct; 3 scorable
  starter_patterns: 7/16 correct; 16 scorable
  starter_transfer: 3/8 correct; 8 scorable


Language evals (untrained): 6/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 1/24 correct; 3 scorable
  starter_patterns: 4/16 correct; 16 scorable
  starter_transfer: 1/8 correct; 8 scorable
Language evals (final): 24/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 0/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable


Language evals (final): 24/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 0/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable
Language evals (final): 24/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 0/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable


Language evals (final): 25/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 2/24 correct; 3 scorable
  starter_patterns: 15/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable
Language evals (final): 26/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 2/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable


Language evals (final): 26/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 2/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable
Language evals (final): 26/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 2/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable


Language evals (final): 26/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 2/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable
Language evals (final): 25/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 1/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable


Language evals (final): 27/48; 27/48 cases have usable vocabulary/context.
  extend_corpus: 3/24 correct; 3 scorable
  starter_patterns: 16/16 correct; 16 scorable
  starter_transfer: 8/8 correct; 8 scorable


| Seed | Baseline correct | Selected correct | Macro loss change | Original guardrail |
|---|---:|---:|---:|---|
| 42 | 24/48 | 26/48 | -0.072404 | True |
| 43 | 26/48 | 26/48 | -0.063569 | True |
| 44 | 25/48 | 27/48 | -0.059996 | True |

## A real update at step 1,000
The passage, prefix, target word and embedding coordinate were selected before training from this update's actual minibatch. The whole network changes during AdamW. The chosen coordinate illustrates that update; its individual causal contribution is not isolated. Report every observed increase and decrease.

In [6]:
update = json.loads((ROOT / study.winner['path'] / 'gradient_update.json').read_text())
print(json.dumps(update, indent=2))

{
  "update": 1000,
  "probe": {
    "document_index": 1859,
    "passage": "a gardener seems calm today .",
    "prefix": "a gardener seems",
    "target": "calm",
    "embedding_token": "gardener",
    "token_id": 101,
    "coordinate": 0,
    "batch_document_indices": [
      4339,
      52,
      3073,
      1982,
      1510,
      2455,
      4047,
      1859,
      4939,
      3090,
      781,
      2407,
      4343,
      1253,
      1705,
      2786,
      2235,
      3499,
      2896,
      3126,
      1313,
      1632,
      3027,
      247,
      1500,
      4206,
      4422,
      2040,
      307,
      3664,
      2326,
      2733
    ]
  },
  "learning_rate": 0.0024088125601003764,
  "before": {
    "batch_loss": 0.6528248190879822,
    "target_probability": 0.21353507041931152,
    "validation": {
      "starter": 0.7884172797203064,
      "grammar": 1.1039743423461914,
      "contrasts": 0.6677941083908081,
      "original": 0.7800863981246948,
      "macro": 0.85339524

## Verification
Recheck frozen sources, split separation, initial weights and minibatches. Reload checkpoints and replay every final evaluation. Reconstruct the measured coordinate update from AdamW moments, learning rate and weight decay.

In [7]:
audit = study.audit()
print('Evidence directory:', output.relative_to(ROOT))

Audit: 44 passed, 0 failed.
Evidence directory: tuning_runs/20260923T065357_257422Z


## Post-run interpretation and next experiments

- Seed 42 selected balanced sampling and peak learning rate 0.003 using validation alone. Category-average validation loss fell from 0.906704 to 0.834300, and later benchmark evaluation improved from 24/48 to 26/48.
- This study changed both initialization and minibatch sampling across seeds. Baseline to selected scores were 24 to 26, 26 to 26, and 25 to 27. Category-average loss fell on all three seeds; original-panel loss increased within the 5% guardrail.
- A separate study held seed-42 minibatches fixed while changing initialization. Its confirmation seeds worsened category-average loss and scored 25 to 25 and 26 to 25. Both studies share the seed-42 run. The candidate remains exploratory; retain baseline settings until the result survives stronger confirmation.
- At the preselected update 1,000, minibatch loss fell from 0.652825 to 0.640562 and P(calm | a gardener seems) rose from 0.213535 to 0.234117. Original-panel validation loss increased. These are effects of the whole network update.
- The independent eight-prompt chat comparison was mixed. Better tense/opposite starts coexisted with malformed continuations and regressions on other prompts.
- Next: correct and broaden original teaching examples, test held-out template families, and cross several initialization and sampling seeds. Coverage remains 27/48; hyperparameters cannot introduce missing words.

[Full report](../../docs/tuning_results.md) · [Companion confirmation study](../../docs/tuning_findings.md) · [Independent chat comparison](../../qa/independent_review_20260923/tuning_chat_followup.md)

These interpretation notes were added after execution. All code outputs remain the recorded execution results.